# European barcode gap list

This notebook builds the European gap list: which European animal species do, and do not yet, have a public DNA barcode.

**How to use it.** Run the sections from top to bottom. To run a cell, click in it and press **Shift + Enter**. Each result starts with **[OK]**, **[WAITING]** or **[PROBLEM]**, followed by what happened and, if needed, what to do.

**You never need to change the code below.** All choices (which countries count as Europe, which data versions to use, and so on) are in the `config` folder.

**Documentation** (in the `docs` folder):
[getting started](../docs/getting-started.md) ·
[how it works](../docs/how-it-works.md) ·
[changing the settings](../docs/settings.md) ·
[reviewing names](../docs/reviewing-names.md) ·
[reading the results](../docs/results.md) ·
[when something goes wrong](../docs/troubleshooting.md)

## 0. Check the settings

Reads the files in the `config` folder, checks them for mistakes and prepares the database. Takes a few seconds.

If something is wrong, the message below names the file and says what to fix. Fix it, save the file, and run this cell again.

In [ ]:
from gaplist.check import run_check

result = run_check()
print(result)

## 1. Build the Catalogue of Life backbone

Gets the Catalogue of Life release named in `config/sources.yaml` and builds the classification of all animal species, with their synonyms. The first time, the download takes a few minutes; building takes about a minute.

Running this again starts a fresh run: results of earlier runs are cleared, because every later step depends on the backbone.

**If ChecklistBank cannot be reached**, download the ColDP zip of the release by hand from https://www.catalogueoflife.org/data/download, put it in the folder `data/col/manual/`, and run this cell again.

In [ ]:
from gaplist.backbone import build_backbone

result = build_backbone()
print(result)
result.by_class.head(20)

## 2. Decide which species occur in Europe

Collects evidence that a species occurs in Europe, and matches every name to Catalogue of Life (including synonyms). Names that cannot be matched with certainty are listed for a person to decide.

### 2a. The earlier European checklist

Reads the earlier gap list (national checklists, Fauna Europaea and others) and matches its names. Takes about a minute.

Names that need a person are listed in `data/review/legacy_names.xlsx`. The first sheet of that file explains what to do. Your decisions go into `config/curation.tsv`; then run this cell again.

In [ ]:
from gaplist.legacy import import_legacy

result = import_legacy()
print(result)
result.outcomes

### 2b. GBIF records

Asks GBIF which animal species have records in the European countries listed in `config/europe.yaml`, using the filters in `config/sources.yaml`. A second, smaller list of records marked as introduced is used to flag species as possibly alien.

**You need a free GBIF account** (https://www.gbif.org). Copy the file `.env.example` to `.env` in the main folder and fill in your GBIF user name, password and e-mail address.

GBIF prepares the lists in the background. The first time you run this cell it will usually say it is waiting: that is normal. Run it again later (from a few minutes to a few hours); it picks up the same lists and does not ask GBIF again.

When it finishes it shows the DOI of the GBIF download. **Cite that DOI** whenever the gap list is published or shared.

In [ ]:
from gaplist.gbif import import_gbif

result = import_gbif()
print(result)
result.summary

## 3. Count barcodes

Counts, for every species in the backbone, the public COI-5P barcodes in a BOLD data package: anywhere in the world, from specimens collected in Europe, and "own" versus "other" (see `config/own_barcodes.yaml`). BOLD specimens collected in Europe also count as evidence that a species occurs there.

**Before running this**, get a BOLD snapshot by hand: log in at BOLD, open the Data Packages page (https://v5.boldsystems.org/?p=21708), download a snapshot (TSV or Parquet), and put the downloaded file, as it is, in the folder `data/bold/manual/`. Keep only one file there.

A full snapshot takes 5 to 15 minutes. BOLD names that could not be matched are listed in `data/review/bold_names.xlsx`, those with the most specimens first: deciding about these can turn gaps into barcoded species. See [reviewing names](../docs/reviewing-names.md).

In [ ]:
from gaplist.bold import import_bold

result = import_bold()
print(result)
result.summary

## 4. Export the target list

Combines the evidence from step 2 and the barcode counts from step 3. A species is on the list when enough sources say it occurs in Europe (`min_sources` in `config/sources.yaml`). How "Wanted" and "Collected" are filled in is set in `config/target_list.yaml`. Takes under a minute.

Writes three files to `data/output/`:

- `targetlist_seed.tsv`: the list in the layout the website loads;
- `gaplist_full.tsv`: the same species with all evidence and barcode details;
- `summary.md`: the data versions used (with DOIs to cite), totals, and gaps per class.

In [ ]:
from gaplist.export import export

result = export()
print(result)
result.by_class.head(20)

### 4b. Update the website

Replaces the species on the target list website with the new list. This needs the website's database details in the `.env` file (ask the target list administrator).

The cell below only **checks** the connection and says what it would change. To really update the website, change `confirm=False` to `confirm=True` and run it again. The update is all or nothing: if anything goes wrong, the website keeps its current list. The website shows the new list after its next refresh (within a few hours).

In [ ]:
from gaplist.deploy import deploy

print(deploy(confirm=False))